# RAG-Fusion + RRF

> Notebook generated from [`examples/rag/07_rag_fusion.py`](../../examples/rag/07_rag_fusion.py).

| Data | Value |
|------|-------|
| **Dataset** | BEIR (embedded, multi-domain IR) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** N generated queries, ranking per query, and RRF score `Σ 1/(k+rank)`.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
RAG-Fusion — Multiple queries + Reciprocal Rank Fusion (RRF)
=============================================================
Architecture: SPEC-RAG-002 / prismal.rag.fusion

Dataset: BEIR (Benchmark for Evaluating IR Models)
  • 18 heterogeneous IR datasets (TREC-COVID, HotpotQA, ArguAna, etc.)
  • Reference: https://huggingface.co/datasets/BeIR/beir
  • Why: RAG-Fusion was specifically designed to improve recall
    by generating multiple variations of the query and fusing their rankings.
    BEIR is the standard benchmark to evaluate retrieval improvements.
    Especially effective on ArguAna (argumentation) and TREC-COVID.

RAG-Fusion architecture description:
  1. Generates N variations of the original query (the LLM paraphrases)
  2. Runs N searches in parallel (one per variation)
  3. Fuses the rankings with RRF (Reciprocal Rank Fusion):
     score(d) = Σ_{q in queries} 1 / (k + rank(d, q))
     where k=60 is the stabilizing parameter (Cormack et al. 2009)
  4. Documents appearing in multiple rankings come out first

  Benefit: Higher recall — a document that does not appear with the original
  query may appear with one of its paraphrases.

Uso:
    uv run python examples/rag/07_rag_fusion.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.documents import Document

from prismal.rag.fusion import FusionResult, RAGFusionEngine, reciprocal_rank_fusion
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: BEIR documents (TREC-COVID + ArguAna style)

In [ ]:
# Scientific and argumentative documents representative of BEIR.
BEIR_DOCUMENTS = [
    # TREC-COVID: papers on COVID-19
    {
        "source": "trec_covid_001",
        "corpus": "TREC-COVID",
        "text": (
            "Effectiveness of COVID-19 mRNA vaccines against the Omicron variant "
            "showed reduced neutralization but maintained protection against severe disease. "
            "Three-dose vaccination regimens provided 70% protection against hospitalization "
            "compared to 40% for two-dose schedules. Booster vaccination timing significantly "
            "impacts efficacy, with optimal boosting at 6-month intervals post-primary series."
        ),
    },
    {
        "source": "trec_covid_002",
        "corpus": "TREC-COVID",
        "text": (
            "Long COVID, also known as post-acute sequelae of SARS-CoV-2 (PASC), affects "
            "approximately 10-30% of infected individuals. Common symptoms include fatigue, "
            "cognitive impairment ('brain fog'), dyspnea, and post-exertional malaise. "
            "Pathophysiological mechanisms include viral persistence, immune dysregulation, "
            "microbiome disruption, and autoimmune responses. No FDA-approved treatments "
            "exist specifically for Long COVID as of 2024."
        ),
    },
    # ArguAna: scientific argumentation
    {
        "source": "arguana_001",
        "corpus": "ArguAna",
        "text": (
            "Universal Basic Income (UBI) represents a transformative approach to social "
            "welfare that provides unconditional regular payments to all citizens. "
            "Proponents argue UBI eliminates poverty traps, reduces bureaucracy of means-tested "
            "welfare, and provides economic security during technological unemployment. "
            "Pilot programs in Finland and Kenya demonstrated improved mental health outcomes "
            "and maintained work incentives contrary to critics' predictions."
        ),
    },
    {
        "source": "arguana_002",
        "corpus": "ArguAna",
        "text": (
            "Artificial intelligence regulation is necessary to prevent monopolistic control "
            "of AI systems by a small number of corporations. Without regulation, AI capabilities "
            "concentrate power among those with computational resources, potentially exacerbating "
            "economic inequality. Regulatory frameworks like the EU AI Act establish risk-based "
            "oversight while preserving innovation. Open-source AI development offers an "
            "alternative path to democratizing AI access."
        ),
    },
    # HotpotQA: multi-hop reasoning documents
    {
        "source": "hotpot_001",
        "corpus": "HotpotQA",
        "text": (
            "The attention mechanism in large language models scales quadratically with "
            "sequence length, creating computational bottlenecks for long documents. "
            "Flash Attention optimizes memory access patterns to achieve linear memory "
            "scaling. Sparse attention variants like Longformer and BigBird extend context "
            "windows to 4096+ tokens using sliding window and global attention patterns."
        ),
    },
    {
        "source": "hotpot_002",
        "corpus": "HotpotQA",
        "text": (
            "Vector databases enable semantic search by storing high-dimensional embeddings "
            "and performing approximate nearest neighbor (ANN) search. Algorithms like "
            "HNSW (Hierarchical Navigable Small World) and IVF (Inverted File Index) "
            "provide sub-linear query time. ChromaDB, Pinecone, Weaviate, and Qdrant "
            "are leading solutions for production RAG deployments requiring low-latency retrieval."
        ),
    },
    {
        "source": "hotpot_003",
        "corpus": "HotpotQA",
        "text": (
            "Retrieval-Augmented Generation improves large language model accuracy by "
            "providing relevant external context at inference time. The retrieval component "
            "uses dense passage retrieval (DPR) or sparse BM25 methods. Advanced RAG "
            "variants include Corrective RAG (CRAG), Self-RAG, RAG-Fusion, and HyDE. "
            "Each variant addresses different failure modes: relevance, hallucination, "
            "query ambiguity, and vocabulary mismatch respectively."
        ),
    },
    {
        "source": "fever_001",
        "corpus": "FEVER",
        "text": (
            "The Transformer architecture was introduced by Vaswani et al. in 2017 in the "
            "landmark paper 'Attention Is All You Need'. Prior to Transformers, sequence "
            "modeling relied on recurrent architectures (LSTM, GRU) that processed tokens "
            "sequentially. Transformers revolutionized NLP by enabling massive parallelization "
            "and direct attention between any two positions in the sequence."
        ),
    },
]

## BEIR queries with multiple equivalent formulations

In [ ]:
BEIR_QUERIES = [
    {
        "id": "BQ1",
        "original_query": "How does Long COVID affect the cognitive system?",
        "corpus": "TREC-COVID",
        "expected_source": "trec_covid_002",
        "expected_queries_contain": ["Long COVID", "cognitive", "brain fog"],
    },
    {
        "id": "BQ2",
        "original_query": "advantages of universal basic income for fighting poverty",
        "corpus": "ArguAna",
        "expected_source": "arguana_001",
        "expected_queries_contain": ["UBI", "welfare", "basic income"],
    },
    {
        "id": "BQ3",
        "original_query": "What vector databases are used in production for RAG?",
        "corpus": "HotpotQA",
        "expected_source": "hotpot_002",
        "expected_queries_contain": ["vector database", "ChromaDB", "Pinecone"],
    },
    {
        "id": "BQ4",
        "original_query": "advanced RAG architectures beyond basic RAG",
        "corpus": "HotpotQA",
        "expected_source": "hotpot_003",
        "expected_queries_contain": ["CRAG", "Self-RAG", "RAG-Fusion", "HyDE"],
    },
]


async def setup_rag_fusion(n_queries: int = 4) -> RAGFusionEngine:
    """Set up RAG-Fusion with the BEIR corpus."""
    store = ChromaVectorStore(collection_name="beir_rag_fusion")

    docs = [
        Document(
            page_content=doc["text"],
            metadata={
                "source": doc["source"],
                "corpus": doc["corpus"],
                "chunk_id": doc["source"],
                "dataset": "BEIR",
            },
        )
        for doc in BEIR_DOCUMENTS
    ]

    store.add_documents(docs)
    print(
        f"  ✓ {len(docs)} BEIR documents indexed ({len({d['corpus'] for d in BEIR_DOCUMENTS})} corpora)"
    )

    return RAGFusionEngine(
        vector_store=store,
        n_queries=n_queries,  # number of query variations to generate
    )


def print_fusion_result(query_info: dict, result: FusionResult) -> None:
    """Show the result with all generated queries and the RRF ranking."""
    print(f"\n[{query_info['id']}] Original query:")
    print(f"  '{query_info['original_query']}'")

    print(f"\n  Queries generated by RAG-Fusion ({len(result.queries)}):")
    for i, q in enumerate(result.queries, 1):
        print(f"    {i}. {q}")

    print("\n  Per-query results:")
    for i, (_query, chunks) in enumerate(
        zip(result.queries, result.per_query_results, strict=False)
    ):
        top_sources = [c.source for c in chunks[:3]]
        print(f"    Q{i + 1}: top-3 → {top_sources}")

    print("\n  Fused ranking (RRF score):")
    for chunk in result.chunks[:5]:
        expected_mark = "→ " if chunk.source == query_info["expected_source"] else "  "
        print(f"    {expected_mark}[{chunk.relevance_score:.4f}] {chunk.source}")

    # Verify recall
    top3_sources = [c.source for c in result.chunks[:3]]
    found = query_info["expected_source"] in top3_sources
    print(f"\n  Expected source in top-3: {'✓' if found else '✗'}")
    print("─" * 70)


def demo_rrf_formula() -> None:
    """Demonstrate the RRF formula with a concrete example."""
    print("\n[RRF Formula — Reciprocal Rank Fusion]")
    print("  score(d) = Σ_{q in queries} 1 / (k + rank(d, q))")
    print("  k = 60 (stabilizing parameter from Cormack et al. 2009)")
    print()

    # Simulated example
    k = 60
    print("  Example with 3 queries and 3 documents:")
    print()
    print(f"  {'Doc':<12} {'Q1 rank':>8} {'Q2 rank':>8} {'Q3 rank':>8} {'RRF score':>12}")
    print("  " + "─" * 52)

    scenarios = [
        ("doc_A", 1, 2, 1),  # ranks high in all → winner
        ("doc_B", 3, 1, 5),  # appears in some
        ("doc_C", 10, 15, 2),  # only strong in Q3
        ("doc_D", None, 3, None),  # only in Q2
    ]

    for doc, r1, r2, r3 in scenarios:
        rrf = 0.0
        if r1:
            rrf += 1 / (k + r1)
        if r2:
            rrf += 1 / (k + r2)
        if r3:
            rrf += 1 / (k + r3)

        r1_str = str(r1) if r1 else "—"
        r2_str = str(r2) if r2 else "—"
        r3_str = str(r3) if r3 else "—"
        print(f"  {doc:<12} {r1_str:>8} {r2_str:>8} {r3_str:>8} {rrf:>12.6f}")

    print()
    print("  Observation: doc_A wins because it ranks first in 2 of 3 queries.")
    print("  doc_C with rank 2 in Q3 beats doc_D with rank 3 in Q2.")

    # Demonstrate the reciprocal_rank_fusion function directly
    from prismal.rag.crag import RetrievedChunk

    sample_lists = [
        [RetrievedChunk("s1", "0", 0.9, "text1"), RetrievedChunk("s2", "1", 0.8, "text2")],
        [RetrievedChunk("s2", "1", 0.7, "text2"), RetrievedChunk("s1", "0", 0.6, "text1")],
    ]
    fused = reciprocal_rank_fusion(sample_lists, k=60)
    print("\n  reciprocal_rank_fusion() demo:")
    for chunk in fused:
        print(f"    {chunk.source}: RRF={chunk.relevance_score:.6f}")


async def main() -> None:
    print("=" * 70)
    print("  RAG-Fusion — Dataset: BEIR (TREC-COVID + ArguAna + HotpotQA)")
    print("=" * 70)

    # Demonstrate the RRF formula
    demo_rrf_formula()

    # Configure engine
    print("\n[RAG-Fusion initialization]")
    engine = await setup_rag_fusion(n_queries=4)
    print("  ✓ RAG-Fusion ready (will generate 4 variations per query)")

    # Run BEIR queries
    print(f"\n[Running {len(BEIR_QUERIES)} BEIR queries]")

    correct_count = 0
    for query_info in BEIR_QUERIES:
        result = await engine.search(query_info["original_query"], k=5)
        print_fusion_result(query_info, result)

        top3 = [c.source for c in result.chunks[:3]]
        if query_info["expected_source"] in top3:
            correct_count += 1

    # Summary
    recall_at_3 = correct_count / len(BEIR_QUERIES)
    print("\n[Summary]")
    print(f"  Recall@3: {correct_count}/{len(BEIR_QUERIES)} ({recall_at_3:.0%})")

    print("\n[Advantages of RAG-Fusion over standard RAG]")
    print("  ✓ Higher recall: a doc not found with query A may be found with B")
    print("  ✓ Robustness: does not depend on the exact wording of the user")
    print("  ✓ No weighting parameters: RRF requires no calibration")
    print("  ✗ Cost: N+1 LLM calls (1 to generate queries + N for searches)")
    print("  ✗ Latency: higher than standard RAG due to N parallel searches")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()